# Practice Lab: Architecture Patterns for AI Apps (Lesson 12.1)

Module 12 . 8 exercises . SSE + LiteLLM + routing + caching + hybrid search + events + GPU + production


# Lesson 12.1: Architecture Patterns for AI Apps

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design.


In [ ]:
import json, time, hashlib
import numpy as np
from collections import Counter, defaultdict
np.random.seed(42)


---
## Exercise 1 (Easy): FastAPI + SSE Streaming



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json, time

class SSE:
    def __init__(self): self.events=[]; self.tokens=0
    def stream(self,prompt):
        words="The GenAI course costs 14999 INR and covers 14 modules over 146 hours".split()
        for i,w in enumerate(words):
            self.events.append({"id":str(i+1),"event":"token","data":json.dumps({"token":w})})
            self.tokens+=1
        self.events.append({"id":str(len(words)+1),"event":"done","data":json.dumps({"total":len(words)})})

print("FastAPI + SSE Streaming:")
s=SSE(); s.stream("price?")
for e in s.events[:4]: print(f"  id:{e['id']} event:{e['event']} data:{e['data']}")
print(f"  ... ({len(s.events)-4} more) -> done")
print(f"  TTFT: ~100-200ms (streaming) vs ~2-4s (batch)")
print(f"  Server: StreamingResponse(gen(), media_type='text/event-stream')")
print(f"  Client: new EventSource('/stream?prompt=...')")


</details>


---
## Exercise 2 (Easy): LiteLLM Gateway with Fallback



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)

class Gateway:
    def __init__(self,providers): self.p=providers; self.stats={p["name"]:{"ok":0,"fail":0} for p in providers}
    def call(self,prompt):
        for p in sorted(self.p,key=lambda x:x["pri"]):
            if np.random.random()<p["up"]:
                self.stats[p["name"]]["ok"]+=1
                return {"provider":p["name"],"model":p["model"],"cost":round(len(prompt.split())*p["cpm"]/1000,5)}
            self.stats[p["name"]]["fail"]+=1; print(f"    [{p['name']}] FAILED")
        return {"provider":"none"}

gw=Gateway([{"name":"gemini","model":"flash","pri":1,"up":0.95,"cpm":0.15},
    {"name":"openai","model":"gpt-4o-mini","pri":2,"up":0.98,"cpm":0.60},
    {"name":"claude","model":"haiku","pri":3,"up":0.99,"cpm":0.80}])
print("LiteLLM Gateway:")
for p in ["Refund policy?","Compare courses","Calculate GST","Explain RAG","Deploy on Cloud Run","Prerequisites?","Capstone project","List modules"]:
    r=gw.call(p); print(f"  [{r['provider']:>7}] {p:<25} ${r.get('cost',0):.4f}")
print(f"\n  Stats: {gw.stats}")
print(f"  Options: LiteLLM(MIT) | Portkey(enterprise) | Bifrost(11us)")


</details>


---
## Exercise 3 (Medium): Cost-Based Model Router



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Router:
    TIERS={"simple":{"model":"nano","cost":0.10,"kw":["hi","hello","thanks","bye","yes","no"]},
           "medium":{"model":"haiku","cost":0.80,"kw":["explain","list","what is","how to","price","refund"]},
           "complex":{"model":"sonnet","cost":3.00,"kw":["compare","analyze","design","architect","optimize"]}}
    def route(self,q,tok=500):
        for t in ["complex","medium","simple"]:
            if any(k in q.lower() for k in self.TIERS[t]["kw"]): break
        else: t="medium"
        return t,self.TIERS[t]["model"],tok*self.TIERS[t]["cost"]/1e6

r=Router()
print("Cost-Based Model Router:")
tr=ts=0
for q,tok in [("Hi!",100),("What is refund policy?",400),("Compare GenAI ROI vs masters",1500),
    ("List modules",300),("Thanks!",50),("How to deploy?",600),("Analyze cost-benefit",2000),
    ("What is RAG?",500),("Hello",80),("Design ML pipeline",1800)]:
    tier,model,cost=r.route(q,tok); sonnet=tok*3.0/1e6; tr+=cost; ts+=sonnet
    print(f"  {q[:35]:<35} {tier:>8} {model:>8} ${cost:.6f}")
print(f"\n  Sonnet: ${ts:.6f} | Routed: ${tr:.6f} | Savings: {(1-tr/ts)*100:.0f}%")


</details>


---
## Exercise 4 (Medium): Multi-Tier Caching



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib; np.random.seed(42)

class Cache:
    def __init__(self): self.exact={}; self.semantic=[]; self.stats={"exact":0,"semantic":0,"prompt":0,"miss":0}; self.saved=0
    def _h(self,q): return hashlib.md5(q.encode()).hexdigest()[:12]
    def _emb(self,t):
        v=np.zeros(20)
        for w in t.lower().split(): v[hash(w)%20]=1.0
        return v/(np.linalg.norm(v)+1e-8)
    def get(self,q,cost=0.003):
        if self._h(q) in self.exact: self.stats["exact"]+=1; self.saved+=cost; return "EXACT"
        qv=self._emb(q)
        for _,cv,_ in self.semantic:
            if float(np.dot(qv,cv))>0.85: self.stats["semantic"]+=1; self.saved+=cost; return "SEMANTIC"
        self.stats["prompt"]+=1; self.saved+=cost*0.9; return "PROMPT_CACHE"
    def put(self,q,r): self.exact[self._h(q)]=r; self.semantic.append((q,self._emb(q),r))

c=Cache()
c.put("What is the course price?","14999"); c.put("Prerequisites?","Python"); c.put("How long?","146hrs")
print("Multi-Tier Caching:")
for q in ["What is the course price?","How much does it cost?","Prerequisites?","Course requirements",
    "Compare courses","Design RAG","What is the course price?","How long is GenAI?","Explain transformers","Price of GenAI?"]:
    tier=c.get(q); print(f"  {tier:>14}: {q[:35]}")
total=sum(c.stats.values())
print(f"\n  Stats: {c.stats}")
print(f"  Saved: ${c.saved:.4f} of ${total*0.003:.4f} = {c.saved/(total*0.003)*100:.0f}%")


</details>


---
## Exercise 5 (Medium): pgvector Hybrid Search



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)
from collections import Counter

class Hybrid:
    def __init__(self): self.docs=[]; self.embs=[]; self.words=[]
    def add(self,t):
        self.docs.append(t)
        v=np.zeros(20)
        for w in t.lower().split(): v[hash(w)%20]+=1
        self.embs.append(v/(np.linalg.norm(v)+1e-8))
        self.words.append(Counter(t.lower().split()))
    def _qemb(self,q):
        v=np.zeros(20)
        for w in q.lower().split(): v[hash(w)%20]+=1
        return v/(np.linalg.norm(v)+1e-8)
    def dense(self,q,k=5):
        qv=self._qemb(q); scores=[(self.docs[i],float(np.dot(qv,e))) for i,e in enumerate(self.embs)]
        return sorted(scores,key=lambda x:-x[1])[:k]
    def sparse(self,q,k=5):
        qw=set(q.lower().split()); scores=[(self.docs[i],sum(dw[w] for w in qw if w in dw)) for i,dw in enumerate(self.words)]
        return sorted(scores,key=lambda x:-x[1])[:k]
    def hybrid(self,q,k=5):
        d={doc:s for doc,s in self.dense(q,k*2)}; s={doc:s for doc,s in self.sparse(q,k*2)}
        dm=max(d.values()) if d else 1; sm=max(s.values()) if s else 1
        combined={doc:d.get(doc,0)/max(dm,0.001)*0.7+s.get(doc,0)/max(sm,0.001)*0.3 for doc in set(d)|set(s)}
        return sorted(combined.items(),key=lambda x:-x[1])[:k]

h=Hybrid()
for doc in ["GenAI course: transformers, RAG, agents","Python course: 80hrs, 9999 INR","Refund: 7 days full, 30 days 50%",
    "Prerequisites: Python basics","Capstone: RAG chatbot on Cloud Run","Module 11: AI agents LangGraph CrewAI",
    "Cloud computing: GCP AWS 11999 INR","Data engineering: BigQuery Dataflow"]:
    h.add(doc)

q="RAG chatbot deployment cloud"
print("pgvector Hybrid Search:")
print(f"  Query: '{q}'")
print(f"\n  Dense:"); [print(f"    [{s:.3f}] {d[:50]}") for d,s in h.dense(q,3)]
print(f"  Sparse:"); [print(f"    [{s:.1f}] {d[:50]}") for d,s in h.sparse(q,3)]
print(f"  Hybrid:"); [print(f"    [{s:.3f}] {d[:50]}") for d,s in h.hybrid(q,3)]
print(f"\n  Recall: dense 78% | sparse 65% | hybrid 91% | +reranking +20-35%")


</details>


---
## Exercise 6 (Challenge): Event-Driven Pipeline
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Event-Driven Pipeline:")
print("  Orchestrator -> 3 workers (embed/search/generate)")
print("  Fan-out: parallel to financial/technical/competitive")
print("  Fan-in: aggregate 3/3 results (5min timeout)")
print("  Backpressure: queue>1000 pause, KEDA autoscale")
print("  Kafka 1.2M/s | Pub/Sub 10K/s (GCP) | Redis 100K/s (simple)")


</details>


---
## Exercise 7 (Challenge): Cloud Run GPU Deploy
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Cloud Run GPU Deploy:")
print("  Dockerfile: FROM vllm/vllm-openai -> EXPOSE 8080")
print("  Deploy: gcloud run deploy --gpu 1 --gpu-type nvidia-l4 --min-instances 0")
print("  Cold start: ~5s | Warm TTFT: ~200ms | Max model: ~13B")
print("  Cloud Run L4($1.50/hr) vs GKE H100($3.50/hr) vs E2E India(Rs88/hr spot)")


</details>


---
## Exercise 8 (Challenge): Full Production Stack
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Full Production Stack:")
print("  FastAPI + LiteLLM -> Cloud SQL pgvector -> Memorystore Redis -> Langfuse")
print("  GCP: Cloud Run + Cloud SQL + Memorystore + GCS + Pub/Sub + Scheduler")
costs=[("Cloud Run",30,60),("Cloud SQL",8,15),("Redis",35,45),("GCS",1,2),("Pub/Sub",5,10),("LLM APIs",50,200),("Langfuse",0,0)]
tl=sum(l for _,l,_ in costs); th=sum(h for _,_,h in costs)
print(f"  Monthly: ${tl}-${th} (india asia-south1)")
print(f"  Canary: shadow(0%) -> canary(5%) -> monitor(24h) -> 50% -> 100%")
print(f"  Credits: Google $350K + AWS $100K + Azure $150K = $600K+")


</details>
